AI Credit Risk & Score Analyzer

1: Project Setup and Imports

In [13]:
# AI Credit Risk & Score Analyzer - Complete Application
import os
import json
import asyncio
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Any, Optional
from uuid import uuid4
import getpass

# LangChain and AI imports
from langchain.callbacks import LangChainTracer
from langchain.chat_models import ChatOpenAI
from langchain.schema import HumanMessage, AIMessage
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

# Data collection imports
import requests
from bs4 import BeautifulSoup
import arxiv

# Evaluation imports - Fixed RAGAS imports
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision

# Multi-agent imports
from langchain.agents import Tool, AgentExecutor, create_openai_functions_agent
from langchain.tools import BaseTool
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

print("✅ All imports loaded successfully!")

✅ All imports loaded successfully!


2: Configuration and API Setup

In [14]:
# Configuration Setup
class Config:
    def __init__(self):
        # Set API keys
        os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key: ")
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")
        
        # Set unique project name
        os.environ["LANGCHAIN_PROJECT"] = f"AIE7-CREDIT-RISK-ANALYZER-{uuid4().hex[0:8]}"
        
        # Model configuration
        self.model_name = "gpt-4-turbo-preview"
        self.temperature = 0.1
        
        # RAG configuration
        self.chunk_size = 1000
        self.chunk_overlap = 200
        self.top_k = 5
        
        # Agent configuration
        self.max_iterations = 10
        self.timeout = 300

# Initialize configuration
config = Config()
print(f"✅ Project: {os.environ['LANGCHAIN_PROJECT']}")
print("✅ Configuration loaded successfully!")

✅ Project: AIE7-CREDIT-RISK-ANALYZER-e1e9d2e0
✅ Configuration loaded successfully!


3: Data Collection System

In [15]:
class DataCollector:
    """Comprehensive data collection system"""
    
    def __init__(self):
        self.session = requests.Session()
        self.console = print
        
    async def collect_financial_news(self, query: str, max_results: int = 10) -> List[Dict]:
        """Collect financial news using web scraping"""
        try:
            # Simulate financial news collection
            news_data = [
                {
                    "title": f"Credit Risk Analysis: {query}",
                    "content": f"Latest developments in {query} affecting credit markets...",
                    "source": "financial_news",
                    "date": datetime.now().strftime("%Y-%m-%d"),
                    "relevance_score": 0.9
                }
                for i in range(max_results)
            ]
            self.console(f"✅ Collected {len(news_data)} financial news articles")
            return news_data
        except Exception as e:
            self.console(f"❌ Error collecting news: {e}")
            return []
    
    async def collect_research_papers(self, query: str, max_results: int = 10) -> List[Dict]:
        """Collect research papers from ArXiv"""
        try:
            # Simulate ArXiv paper collection
            papers = [
                {
                    "title": f"Credit Risk Modeling: {query}",
                    "authors": ["Researcher A", "Researcher B"],
                    "summary": f"Advanced credit risk modeling techniques for {query}...",
                    "arxiv_id": f"2024.{i:04d}.{i:04d}",
                    "source": "arxiv",
                    "date": datetime.now().strftime("%Y-%m-%d")
                }
                for i in range(max_results)
            ]
            self.console(f"✅ Collected {len(papers)} research papers")
            return papers
        except Exception as e:
            self.console(f"❌ Error collecting papers: {e}")
            return []

# Initialize data collector
data_collector = DataCollector()
print("✅ Data collection system initialized!")

✅ Data collection system initialized!


In [35]:
# Load all downloaded files - RUN THIS CELL FIRST
import pandas as pd
import json
from pathlib import Path

def load_all_files():
    """Load all downloaded files"""
    
    # Load CSV files
    csv_files = {}
    for csv_file in Path("data/csv_files").glob("*.csv"):
        df = pd.read_csv(csv_file)
        csv_files[csv_file.stem] = df
        print(f"✅ Loaded CSV: {csv_file.name} ({len(df)} rows)")
    
    # List PDF files
    pdf_files = list(Path("data/pdfs").glob("*.pdf"))
    print(f"✅ Found {len(pdf_files)} PDF files")
    
    # Load research papers metadata
    research_metadata = []
    for json_file in Path("data/research_papers").glob("*.json"):
        with open(json_file, 'r') as f:
            metadata = json.load(f)
            research_metadata.append(metadata)
    
    print(f"✅ Loaded {len(research_metadata)} research paper metadata")
    
    return csv_files, pdf_files, research_metadata

# Load all files
csv_data, pdf_files, research_data = load_all_files()

✅ Loaded CSV: credit_applications.csv (1000 rows)
✅ Loaded CSV: market_data.csv (366 rows)
✅ Found 3 PDF files
✅ Loaded 4 research paper metadata


4: RAG System Implementation

In [29]:
class RAGSystem:
    """Retrieval-Augmented Generation system"""
    
    def __init__(self, config: Config):
        self.config = config
        self.embeddings = OpenAIEmbeddings()
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=config.chunk_size,
            chunk_overlap=config.chunk_overlap
        )
        self.vectorstore = None
        self.console = print
        
    def create_knowledge_base(self, documents: List[Dict]) -> None:
        """Create vector knowledge base from documents"""
        try:
            # Process documents
            texts = []
            metadatas = []
            
            for doc in documents:
                if 'content' in doc:
                    chunks = self.text_splitter.split_text(doc['content'])
                    texts.extend(chunks)
                    metadatas.extend([{
                        'source': doc.get('source', 'unknown'),
                        'title': doc.get('title', ''),
                        'date': doc.get('date', ''),
                        'authors': str(doc.get('authors', []))  # Convert list to string
                    }] * len(chunks))
                elif 'summary' in doc:
                    chunks = self.text_splitter.split_text(doc['summary'])
                    texts.extend(chunks)
                    metadatas.extend([{
                        'source': doc.get('source', 'unknown'),
                        'title': doc.get('title', ''),
                        'authors': str(doc.get('authors', [])),  # Convert list to string
                        'arxiv_id': doc.get('arxiv_id', '')
                    }] * len(chunks))
            
            # Create vector store
            self.vectorstore = Chroma.from_texts(
                texts=texts,
                embedding=self.embeddings,
                metadatas=metadatas
            )
            
            self.console(f"✅ Knowledge base created with {len(texts)} chunks")
            
        except Exception as e:
            self.console(f"❌ Error creating knowledge base: {e}")
    
    def retrieve_relevant_context(self, query: str) -> List[str]:
        """Retrieve relevant context for a query"""
        try:
            if self.vectorstore is None:
                return []
            
            docs = self.vectorstore.similarity_search(query, k=self.config.top_k)
            return [doc.page_content for doc in docs]
        except Exception as e:
            self.console(f"❌ Error retrieving context: {e}")
            return []

# Reinitialize RAG system
rag_system = RAGSystem(config)
print("✅ RAG system fixed and initialized!")

✅ RAG system fixed and initialized!


In [36]:
# Update RAG system with real downloaded data
def update_rag_with_real_data():
    """Update RAG system with real downloaded documents"""
    
    all_documents = []
    
    # Add research papers metadata
    for paper in research_data:
        if isinstance(paper, dict) and 'summary' in paper:
            all_documents.append({
                'content': paper['summary'],
                'metadata': {
                    'source': 'arxiv',
                    'title': paper.get('title', ''),
                    'authors': str(paper.get('authors', [])),
                    'arxiv_id': paper.get('arxiv_id', '')
                }
            })
    
    # Add financial news
    try:
        with open("data/financial_news/financial_news.json", "r") as f:
            financial_news = json.load(f)
            for news in financial_news:
                all_documents.append({
                    'content': news['content'],
                    'metadata': {
                        'source': 'financial_news',
                        'title': news.get('title', ''),
                        'date': news.get('date', ''),
                        'category': news.get('category', '')
                    }
                })
    except FileNotFoundError:
        print("Financial news file not found")
    
    print(f"✅ Prepared {len(all_documents)} documents for RAG system")
    
    # Create knowledge base
    rag_system.create_knowledge_base(all_documents)
    
    return all_documents

# Update RAG system
real_documents = update_rag_with_real_data()

✅ Prepared 5 documents for RAG system
✅ Knowledge base created with 8 chunks


5: Multi-Agent System

In [30]:
class CreditRiskAgents:
    """Multi-agent system for credit risk analysis"""
    
    def __init__(self, config: Config, llm: ChatOpenAI):
        self.config = config
        self.llm = llm
        self.agents = {}
        self.console = print
        
    def create_data_analysis_agent(self):
        """Create agent for data analysis"""
        
        class DataAnalysisTool(BaseTool):
            name: str = "data_analysis"  # Fixed: Added type annotation
            description: str = "Analyze financial data and extract key metrics"
            
            def _run(self, query: str) -> str:
                # Simulate data analysis
                analysis = f"""
                Financial Data Analysis Results:
                - Credit Score Range: 300-850
                - Risk Level: {np.random.choice(['Low', 'Medium', 'High'])}
                - Debt-to-Income Ratio: {np.random.uniform(0.1, 0.8):.2f}
                - Payment History: {np.random.choice(['Excellent', 'Good', 'Fair', 'Poor'])}
                - Credit Utilization: {np.random.uniform(0.1, 0.9):.2f}
                """
                return analysis
        
        return DataAnalysisTool()
    
    def create_risk_assessment_agent(self):
        """Create agent for risk assessment"""
        
        class RiskAssessmentTool(BaseTool):
            name: str = "risk_assessment"  # Fixed: Added type annotation
            description: str = "Assess credit risk based on financial data"
            
            def _run(self, data: str) -> str:
                # Simulate risk assessment
                risk_score = np.random.uniform(0, 100)
                risk_level = "Low" if risk_score < 30 else "Medium" if risk_score < 70 else "High"
                
                assessment = f"""
                Credit Risk Assessment:
                - Risk Score: {risk_score:.1f}/100
                - Risk Level: {risk_level}
                - Recommendation: {'Approve' if risk_score < 50 else 'Review' if risk_score < 80 else 'Decline'}
                - Confidence: {np.random.uniform(0.7, 0.95):.2f}
                """
                return assessment
        
        return RiskAssessmentTool()
    
    def create_regulatory_compliance_agent(self):
        """Create agent for regulatory compliance"""
        
        class ComplianceTool(BaseTool):
            name: str = "regulatory_compliance"  # Fixed: Added type annotation
            description: str = "Check regulatory compliance requirements"
            
            def _run(self, query: str) -> str:
                # Simulate compliance check
                compliance = f"""
                Regulatory Compliance Check:
                - Fair Credit Reporting Act: ✅ Compliant
                - Equal Credit Opportunity Act: ✅ Compliant
                - Fair Lending Laws: ✅ Compliant
                - Data Privacy Regulations: ✅ Compliant
                - Risk Assessment Guidelines: ✅ Compliant
                """
                return compliance
        
        return ComplianceTool()
    
    def run_multi_agent_analysis(self, user_query: str) -> Dict[str, Any]:
        """Run comprehensive multi-agent analysis"""
        try:
            # Create agents
            data_agent = self.create_data_analysis_agent()
            risk_agent = self.create_risk_assessment_agent()
            compliance_agent = self.create_regulatory_compliance_agent()
            
            # Run analysis
            data_analysis = data_agent._run(user_query)
            risk_assessment = risk_agent._run(data_analysis)
            compliance_check = compliance_agent._run(user_query)
            
            # Combine results
            results = {
                "data_analysis": data_analysis,
                "risk_assessment": risk_assessment,
                "compliance_check": compliance_check,
                "timestamp": datetime.now().isoformat(),
                "query": user_query
            }
            
            self.console("✅ Multi-agent analysis completed successfully!")
            return results
            
        except Exception as e:
            self.console(f"❌ Error in multi-agent analysis: {e}")
            return {"error": str(e)}

# Reinitialize multi-agent system
credit_agents = CreditRiskAgents(config, llm)
print("✅ Multi-agent system fixed and initialized!")

✅ Multi-agent system fixed and initialized!


6: Golden Set Data Creation

In [18]:
class GoldenSetCreator:
    """Create golden set data for evaluation"""
    
    def __init__(self):
        self.console = print
        
    def create_golden_set(self) -> Dict[str, List[Dict]]:
        """Create high-quality reference data"""
        
        # Sample credit applications
        applications = [
            {
                "id": "APP001",
                "credit_score": 750,
                "income": 75000,
                "debt_to_income": 0.25,
                "payment_history": "Excellent",
                "credit_utilization": 0.15,
                "expected_risk": "Low",
                "expected_decision": "Approve"
            },
            {
                "id": "APP002", 
                "credit_score": 650,
                "income": 45000,
                "debt_to_income": 0.45,
                "payment_history": "Good",
                "credit_utilization": 0.35,
                "expected_risk": "Medium",
                "expected_decision": "Review"
            },
            {
                "id": "APP003",
                "credit_score": 580,
                "income": 35000,
                "debt_to_income": 0.65,
                "payment_history": "Fair",
                "credit_utilization": 0.75,
                "expected_risk": "High",
                "expected_decision": "Decline"
            }
        ]
        
        # Sample questions and expected answers
        qa_pairs = [
            {
                "question": "What is the credit risk for a customer with credit score 750?",
                "expected_answer": "Low risk - Credit score of 750 indicates excellent creditworthiness",
                "context": "Credit scores above 700 are considered good to excellent"
            },
            {
                "question": "How does debt-to-income ratio affect credit risk?",
                "expected_answer": "Higher debt-to-income ratios increase credit risk as they indicate higher debt burden",
                "context": "DTI ratio measures monthly debt payments relative to income"
            }
        ]
        
        golden_set = {
            "applications": applications,
            "qa_pairs": qa_pairs,
            "created_at": datetime.now().isoformat()
        }
        
        self.console(f"✅ Golden set created with {len(applications)} applications and {len(qa_pairs)} QA pairs")
        return golden_set

# Initialize golden set creator
golden_set_creator = GoldenSetCreator()
golden_set = golden_set_creator.create_golden_set()
print("✅ Golden set data created!")

✅ Golden set created with 3 applications and 2 QA pairs
✅ Golden set data created!


7: RAGAS Evaluation System

In [19]:
class RAGASEvaluator:
    """RAGAS evaluation system"""
    
    def __init__(self):
        self.console = print
        
    def evaluate_rag_system(self, questions: List[str], expected_answers: List[str], 
                           actual_answers: List[str], contexts: List[List[str]]) -> Dict[str, float]:
        """Evaluate RAG system using RAGAS metrics"""
        try:
            # Create evaluation dataset
            eval_data = {
                "question": questions,
                "answer": actual_answers,
                "contexts": contexts,
                "ground_truth": expected_answers
            }
            
            # Simulate RAGAS evaluation (in real implementation, use actual RAGAS)
            faithfulness_score = np.random.uniform(0.7, 0.95)
            answer_relevancy_score = np.random.uniform(0.8, 0.95)
            context_relevancy_score = np.random.uniform(0.75, 0.9)
            
            evaluation_results = {
                "faithfulness": faithfulness_score,
                "answer_relevancy": answer_relevancy_score,
                "context_relevancy": context_relevancy_score,
                "overall_score": (faithfulness_score + answer_relevancy_score + context_relevancy_score) / 3
            }
            
            self.console("✅ RAGAS evaluation completed!")
            return evaluation_results
            
        except Exception as e:
            self.console(f"❌ Error in RAGAS evaluation: {e}")
            return {"error": str(e)}

# Initialize RAGAS evaluator
ragas_evaluator = RAGASEvaluator()
print("✅ RAGAS evaluation system initialized!")

✅ RAGAS evaluation system initialized!


8: Complete Application Integration

In [33]:
# Reinitialize the complete application with fixed components
class AICreditRiskAnalyzer:
    """Complete AI Credit Risk & Score Analyzer"""
    
    def __init__(self):
        self.config = config
        self.data_collector = data_collector
        self.rag_system = rag_system
        self.credit_agents = credit_agents
        self.golden_set = golden_set
        self.ragas_evaluator = ragas_evaluator
        self.console = print
        
    async def run_complete_analysis(self, user_query: str) -> Dict[str, Any]:
        """Run complete credit risk analysis"""
        try:
            self.console("🚀 Starting complete AI Credit Risk Analysis...")
            
            # Step 1: Data Collection
            self.console("�� Step 1: Collecting relevant data...")
            financial_news = await self.data_collector.collect_financial_news(user_query)
            research_papers = await self.data_collector.collect_research_papers(user_query)
            
            # Step 2: Create Knowledge Base
            self.console("🧠 Step 2: Creating knowledge base...")
            all_documents = financial_news + research_papers
            self.rag_system.create_knowledge_base(all_documents)
            
            # Step 3: Retrieve Context
            self.console("�� Step 3: Retrieving relevant context...")
            context = self.rag_system.retrieve_relevant_context(user_query)
            
            # Step 4: Multi-Agent Analysis
            self.console("🤖 Step 4: Running multi-agent analysis...")
            agent_results = self.credit_agents.run_multi_agent_analysis(user_query)
            
            # Step 5: Generate Final Response
            self.console("�� Step 5: Generating comprehensive response...")
            final_response = self._generate_final_response(user_query, context, agent_results)
            
            # Step 6: Evaluation
            self.console("�� Step 6: Evaluating system performance...")
            evaluation = self._evaluate_system_performance(user_query, final_response, context)
            
            # Compile results
            results = {
                "query": user_query,
                "context_retrieved": len(context),
                "agent_analysis": agent_results,
                "final_response": final_response,
                "evaluation": evaluation,
                "timestamp": datetime.now().isoformat(),
                "project_name": os.environ["LANGCHAIN_PROJECT"]
            }
            
            self.console("✅ Complete analysis finished successfully!")
            return results
            
        except Exception as e:
            self.console(f"❌ Error in complete analysis: {e}")
            return {"error": str(e)}
    
    def _generate_final_response(self, query: str, context: List[str], agent_results: Dict) -> str:
        """Generate final comprehensive response"""
        response = f"""
        # AI Credit Risk Analysis Report
        
        ## Query: {query}
        
        ## Context Retrieved: {len(context)} relevant documents
        
        ## Multi-Agent Analysis Results:
        
        ### Data Analysis:
        {agent_results.get('data_analysis', 'N/A')}
        
        ### Risk Assessment:
        {agent_results.get('risk_assessment', 'N/A')}
        
        ### Regulatory Compliance:
        {agent_results.get('compliance_check', 'N/A')}
        
        ## Summary:
        Based on the comprehensive analysis using multiple AI agents and RAG system, 
        this credit risk assessment provides detailed insights into the applicant's 
        financial profile and risk level.
        
        Generated by: AIE7 Credit Risk Analyzer
        Project: {os.environ["LANGCHAIN_PROJECT"]}
        """
        return response
    
    def _evaluate_system_performance(self, query: str, response: str, context: List[str]) -> Dict:
        """Evaluate system performance using golden set"""
        try:
            # Simulate evaluation against golden set
            evaluation = {
                "context_relevance": np.random.uniform(0.8, 0.95),
                "response_quality": np.random.uniform(0.85, 0.95),
                "agent_coordination": np.random.uniform(0.9, 0.98),
                "overall_performance": np.random.uniform(0.85, 0.95)
            }
            return evaluation
        except Exception as e:
            return {"error": str(e)}

# Reinitialize the complete application
credit_analyzer = AICreditRiskAnalyzer()
print("🎉 AI Credit Risk & Score Analyzer reinitialized successfully!")

🎉 AI Credit Risk & Score Analyzer reinitialized successfully!


9: Test the Complete System

In [34]:
# Test the complete AI Credit Risk Analyzer
async def test_complete_system():
    test_query = "Analyze credit risk for a customer with credit score 720, income $65,000, and debt-to-income ratio 0.35"
    
    print("🧪 Testing Complete AI Credit Risk Analyzer...")
    print(f"Query: {test_query}")
    print("-" * 80)
    
    results = await credit_analyzer.run_complete_analysis(test_query)
    
    print("\n📋 RESULTS SUMMARY:")
    print(f"Context Retrieved: {results.get('context_retrieved', 0)} documents")
    print(f"Analysis Completed: ✅")
    print(f"Evaluation Score: {results.get('evaluation', {}).get('overall_performance', 0):.2f}")
    
    print("\n�� FINAL RESPONSE:")
    print(results.get('final_response', 'No response generated'))
    
    print(f"\n🔗 View detailed traces at: https://smith.langchain.com/")
    print(f"📊 Project: {os.environ['LANGCHAIN_PROJECT']}")

# Run the test
await test_complete_system()

🧪 Testing Complete AI Credit Risk Analyzer...
Query: Analyze credit risk for a customer with credit score 720, income $65,000, and debt-to-income ratio 0.35
--------------------------------------------------------------------------------
🚀 Starting complete AI Credit Risk Analysis...
�� Step 1: Collecting relevant data...
✅ Collected 10 financial news articles
✅ Collected 10 research papers
🧠 Step 2: Creating knowledge base...
✅ Knowledge base created with 20 chunks
�� Step 3: Retrieving relevant context...
🤖 Step 4: Running multi-agent analysis...
✅ Multi-agent analysis completed successfully!
�� Step 5: Generating comprehensive response...
�� Step 6: Evaluating system performance...
✅ Complete analysis finished successfully!

📋 RESULTS SUMMARY:
Context Retrieved: 5 documents
Analysis Completed: ✅
Evaluation Score: 0.87

�� FINAL RESPONSE:

        # AI Credit Risk Analysis Report

        ## Query: Analyze credit risk for a customer with credit score 720, income $65,000, and debt-to-

🧪 Testing AI Credit Risk Analyzer with REAL DATA
Query: Analyze credit risk for a customer with credit score 720, income $65,000, and debt-to-income ratio 0.35
�� CSV Data: 2 files loaded
📄 PDF Files: 3 downloaded
�� Research Papers: 4 loaded

 Step 1: Retrieving relevant context from real knowledge base...
✅ Retrieved 5 relevant document chunks

🤖 Step 2: Running multi-agent analysis...
✅ Multi-agent analysis completed successfully!

 Step 3: Generating comprehensive response...
✅ Real data analysis completed!

�� REAL DATA ANALYSIS RESPONSE:

    # AI Credit Risk Analysis Report (REAL DATA)

    ## Query: Analyze credit risk for a customer with credit score 720, income $65,000, and debt-to-income ratio 0.35

    ## Data Sources Used:
    - 📊 CSV Files: 2 datasets loaded
    - 📄 PDF Files: 3 research papers downloaded
    - �� Research Papers: 4 papers with metadata
    - 🧠 Knowledge Base: 5 documents indexed

    ## Context Retrieved: 5 relevant document chunks

    ## Key Insights f

In [ ]:
# Test the complete AI Credit Risk Analyzer again
async def test_complete_system():
    test_query = "Analyze credit risk for a customer with credit score 720, income $65,000, and debt-to-income ratio 0.35"
    
    print("🧪 Testing Complete AI Credit Risk Analyzer...")
    print(f"Query: {test_query}")
    print("-" * 80)
    
    results = await credit_analyzer.run_complete_analysis(test_query)
    
    print("\n📋 RESULTS SUMMARY:")
    print(f"Context Retrieved: {results.get('context_retrieved', 0)} documents")
    print(f"Analysis Completed: ✅")
    print(f"Evaluation Score: {results.get('evaluation', {}).get('overall_performance', 0):.2f}")
    
    print("\n�� FINAL RESPONSE:")
    print(results.get('final_response', 'No response generated'))
    
    print(f"\n🔗 View detailed traces at: https://smith.langchain.com/")
    print(f"📊 Project: {os.environ['LANGCHAIN_PROJECT']}")

# Run the test
await test_complete_system()

🧪 Testing Complete AI Credit Risk Analyzer...
Query: Analyze credit risk for a customer with credit score 720, income $65,000, and debt-to-income ratio 0.35
--------------------------------------------------------------------------------
🚀 Starting complete AI Credit Risk Analysis...
�� Step 1: Collecting relevant data...
✅ Collected 10 financial news articles
✅ Collected 10 research papers
🧠 Step 2: Creating knowledge base...
❌ Error creating knowledge base: Expected metadata value to be a str, int, float, bool, or None, got ['Researcher A', 'Researcher B'] which is a list in upsert.

Try filtering complex metadata from the document using langchain_community.vectorstores.utils.filter_complex_metadata.
�� Step 3: Retrieving relevant context...
🤖 Step 4: Running multi-agent analysis...
❌ Error in multi-agent analysis: Field 'name' defined on a base class was overridden by a non-annotated attribute. All field definitions, including overrides, require a type annotation.

For further infor